# 07 — Multi-Disease Dataset Plan

This notebook records the disease-extension roadmap for the thesis and the results of
the dataset investigation conducted for the planned PNEUMONIA class extension.

All claims are based on verified public information. No class is committed to the
pipeline until the corresponding dataset has been downloaded, audited, and confirmed
to have explicit, unambiguous labels in cough audio.

## Current implemented scope

Binary screening:

- `COVID`
- `HEALTHY_OR_NONTARGET`

Datasets in use:

- **Coswara** — Zenodo record 7188627, CC-BY 4.0, cough-only recordings with COVID status labels.
- **COUGHVID** — Zenodo record 4498364, CC-BY 4.0, crowdsourced cough recordings with physician-annotated labels.

Label mapping rules are codified in `src/config/label_schema.py`. `HEALTHY_OR_NONTARGET`
is a **mixed control class** (healthy + non-target controls), documented as a limitation
in the thesis report.

**Current label distribution** (unified metadata, `metadata/cleaned/unified_metadata.csv`):

| Class | Samples |
|---|---|
| HEALTHY_OR_NONTARGET | 16,132 |
| COVID | 2,517 |

Neither existing dataset contains explicit PNEUMONIA, TB, or LRTI-subcategory labels.

## Target disease extension: PNEUMONIA

The supervisor has directed that PNEUMONIA be the next candidate extension class,
superseding the previous general `OTHER_RESPIRATORY_ILLNESS` placeholder.

**Rationale for PNEUMONIA as the next target:**
- Pneumonia is a major, clinically high-priority respiratory disease.
- It is strongly associated with cough as a primary symptom.
- Unlike asthma (see below), cough recordings from pneumonia patients exist in
  purpose-built public datasets.
- It aligns with the thesis framing of rapid, non-invasive respiratory screening.

**Constraint:** PNEUMONIA will only be added if a dataset with an **explicit, per-sample
PNEUMONIA label in cough audio** can be obtained and audited. Generic respiratory
illness labels or inferred labels are not acceptable.

## Dataset investigation: Sound-Dr

### What was investigated

Sound-Dr (Hoang et al., 2022 — arXiv:2201.04581, GitHub: ReML-AI/Sound-Dr) was the
primary dataset under investigation.

**Verified facts:**

| Property | Finding |
|---|---|
| Full name | Sound-Dr: Reliable Sound Dataset and Baseline AI System for Respiratory Illnesses |
| Creators | Hoang et al., FPT Software / Vietnam National University, 2022 |
| Paper | arXiv:2201.04581 |
| Audio types | Coughing, **mouth breathing**, **nose breathing** (three types per subject) |
| Conditions mentioned in abstract | COVID-19, pneumonia, respiratory illnesses |
| Approximate subjects | ~1,310 subjects (26 % COVID-positive, 74 % COVID-negative) |
| Licence for dataset | **Not clearly stated** — code repo is public but dataset license documentation is absent from available sources |
| Download mechanism | **Not confirmed** — no Zenodo/FigShare mirror found; GitHub repository access method not documented |
| Pneumonia as explicit label | **Unconfirmed** — abstract mentions pneumonia as a condition; per-sample label schema not retrievable from public sources |
| Cough-only filtering | **Not possible as default** — recordings include mouth-breathing and nose-breathing alongside cough |

### Verdict on Sound-Dr

**Sound-Dr is not yet integrable** for the following reasons:

1. **Mixed modalities.** Each recording session includes coughing, mouth breathing, and
   nose breathing. The Coswara and COUGHVID training corpus contains cough-only
   recordings. Adding breathing sounds would introduce a modality mismatch unless
   each file type is explicitly filtered to cough-only before mixing with the existing data.

2. **Unconfirmed download access.** The GitHub repository's data access path could not
   be verified from public sources. Dataset integration requires a confirmed, documented
   download procedure.

3. **Unclear per-sample label schema.** Whether each audio file carries an individual
   `PNEUMONIA` label or only a subject-level diagnosis record is not confirmed.

4. **Unclear dataset licence.** Thesis-use and redistribution rights must be confirmed
   before any data is downloaded or included.

**Action required before Sound-Dr can be used:**

- Visit the GitHub repository (https://github.com/ReML-AI/Sound-Dr) directly in a browser.
- Confirm the data download link or request procedure.
- Read the data README to verify: (a) explicit per-sample PNEUMONIA label exists,
  (b) cough files can be isolated from breathing files by filename or metadata,
  (c) licence permits thesis use.
- If all three are satisfied, add a dataset registry entry in `src/data/dataset_registry.py`
  and a label mapping function in `src/config/label_schema.py`.

## Other pneumonia cough datasets found

| Dataset | Explicit PNEUMONIA label | Cough-only | Open access | Size | Notes |
|---|---|---|---|---|---|
| **CFCS (FigShare)** | Yes — pediatric bronchitis vs pneumonia | Yes | Needs verification (returned 403 during automated check) | 82 pneumonia, 91 bronchitis samples | Pediatric (ages 0–11) only; small; may not generalise |
| **COUGHVID** (already in use) | No — LRTI (bronchitis, pneumonia, asthma grouped) | Yes | Yes (Zenodo) | ~2,800 expert-labelled | Pneumonia is not separately extractable; LRTI is a superset |
| **Coswara** (already in use) | No — COVID / healthy labels only | Yes | Yes (Zenodo) | 5,015 in current metadata | No LRTI or pneumonia column |
| **NoCoCoDa** | No | Yes | Yes | 73 cough events | COVID-only |
| **CODA TB** | No | Yes | Controlled access | 700,000+ coughs | TB vs non-TB; no pneumonia sub-label |

**Key finding:** No openly downloadable, cough-only, adult-population pneumonia dataset with
explicit per-sample labels was confirmed as immediately integrable. The CFCS dataset is
the only identified source with explicit pneumonia cough labels, but it is pediatric-only
and very small (82 samples), which would create a severe class-size imbalance against
2,517 COVID and 16,132 HEALTHY samples in the current training corpus.

## Why asthma is intentionally NOT included

Most freely available asthma datasets are **lung auscultation recordings** (stethoscope
placed on the chest wall), not mobile-device cough recordings. Mixing auscultation with
cough data introduces a **modality mismatch**: the model would learn the microphone
placement rather than disease-specific cough characteristics.

Asthma will only be reconsidered if a clean, cough-only, explicitly-labelled asthma
dataset with a recording protocol comparable to Coswara or COUGHVID is found.

## Tuberculosis — pending controlled access

The CODA / UCSF TB dataset (https://tbdata.ucsf.edu, Synapse `syn31472953`) requires
registration and approval. The pipeline already contains placeholder infrastructure:

- `data/raw/tb/README_MANUAL_DOWNLOAD.md`
- TB label rules in `src/config/label_schema.py`
- TB metadata audit slot in `metadata/raw/tb_audit.json`

TB remains on the roadmap and takes priority over PNEUMONIA once access is granted,
because the dataset is already known and the pipeline is pre-wired for it.

## Decision: current implementation stays binary

Based on the dataset investigation:

- Sound-Dr cannot be integrated until download access, licence, per-sample label
  schema, and cough-file isolation are all confirmed.
- No other adult-population, open-access, cough-only pneumonia dataset was identified
  as immediately integrable.
- The only confirmed explicit-pneumonia cough dataset (CFCS) is pediatric and too small
  to train a meaningful third class.

**The project remains binary (COVID vs HEALTHY_OR_NONTARGET) until one of the
following conditions is met:**

1. Sound-Dr download, licence, and per-sample PNEUMONIA labels are confirmed, AND
   cough files can be isolated from breathing files.
2. Another explicitly-labelled adult pneumonia cough dataset is found and verified.
3. TB controlled-access data arrives (TB takes priority).

This decision is documented here so it can be cited in the thesis Discussion section
as a deliberate, evidence-based limitation rather than an oversight.

## Final class roadmap

| Stage | Classes | Condition |
|---|---|---|
| **Current** | COVID, HEALTHY_OR_NONTARGET | Implemented |
| **Next (pneumonia path)** | COVID, PNEUMONIA, HEALTHY_OR_NONTARGET | Requires Sound-Dr (or equivalent) access + cough isolation + explicit label confirmed |
| **Next (TB path)** | COVID, TB, HEALTHY_OR_NONTARGET | Requires CODA/UCSF access approval |
| **Final intended** | COVID, PNEUMONIA, TB, HEALTHY_OR_NONTARGET | Both above conditions met |

Asthma is **not** in any stage of this roadmap.  
Generic LRTI labels from COUGHVID are **not** mapped to PNEUMONIA unless sub-labels are confirmed.